# 05 — The model and the loss

**What this notebook does.** Builds the U-Net that predicts boundary logits,
proves that its first convolution really did inherit the pretrained RGB filters
as a sum rather than losing them, measures how large a batch fits at patch 256
*with the optimizer state*, and then takes the compound loss apart on real
tiles — showing what each of `BCE`, `Dice` and `clDice` charges for a broken
line, a fattened line, and a prediction of nothing at all.

**What must already exist.**

- `GH_TOKEN` as a host secret and the datasets mounted — `00_bootstrap.ipynb`
- `reports/manifests/*.csv` and `configs/fold_stats.yaml` — `03_tiling.ipynb`
- `reports/gt_extraction.json` and the boundary PNGs — `02_boundary_gt.ipynb`
- `configs/dataloader.yaml` — `04_dataset.ipynb` (not required, but the
  persistent tile cache it wrote makes the loss demo instant instead of slow)

**What it produces.** `train.batch_size` in `configs/default.yaml` — the
largest batch that fits on this host at patch 256 including Adam's state — and
the evidence that the loss behaves the way step 6 is going to assume it does.
No model is trained and no checkpoint is written.

**Expected runtime on a free T4.** 6–10 minutes. Most of it is the batch-size
sweep; the encoder weights are downloaded once (~85 MB) on a cold session.

## Cell 1 — the standard bootstrap block

Identical in every notebook. Reads `GH_TOKEN` from the host secret store,
fetches `scripts/bootstrap_session.py`, then hands over to `bootstrap()`,
which syncs the repo, installs what is missing, mounts Drive on Colab and
returns `PATHS`.

In [ ]:
# --- standard bootstrap block: identical in every notebook ---------------
OWNER, REPO, BRANCH = "arhorri", "boundary", "main"

import importlib, os, pathlib, sys, urllib.request


def _gh_token():
    """Read GH_TOKEN from whichever secret store this host provides."""
    try:
        from google.colab import userdata

        return userdata.get("GH_TOKEN")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    return os.environ.get("GH_TOKEN")


_token = _gh_token()
if not _token:
    raise SystemExit(
        "GH_TOKEN secret is missing.\n"
        "  Colab : key icon in the left sidebar -> add GH_TOKEN -> notebook access ON\n"
        "  Kaggle: Add-ons -> Secrets -> add GH_TOKEN -> attach to this notebook"
    )

_req = urllib.request.Request(
    f"https://api.github.com/repos/{OWNER}/{REPO}/contents/scripts/bootstrap_session.py?ref={BRANCH}",
    headers={
        "Authorization": f"Bearer {_token}",
        "Accept": "application/vnd.github.raw",
    },
)
pathlib.Path("bootstrap_session.py").write_bytes(urllib.request.urlopen(_req).read())
del _token

if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))
import bootstrap_session

bootstrap_session = importlib.reload(bootstrap_session)

PATHS = bootstrap_session.bootstrap(
    repo_url=f"https://github.com/{OWNER}/{REPO}.git", branch=BRANCH
)

## Why the first convolution has to be adapted, not replaced

The encoder is a ResNet-34 pretrained on ImageNet, which is three-channel RGB.
The micrographs are one channel. Something has to happen to that first
`7x7x3x64` convolution, and there are three candidates:

| what you could do | what it costs |
| --- | --- |
| take one of the three channels | throws away two thirds of the learned evidence |
| reinitialise the layer randomly | throws away the pretraining, which was the entire reason to use this encoder |
| **sum the three channels** | keeps it |

Summing is right because of what these filters *are*. An early ImageNet filter
is an oriented edge or blob detector; it fires on a luminance step. In RGB that
step usually appears in all three channels at once, so `w_R * I + w_G * I +
w_B * I = (w_R + w_G + w_B) * I` — the summed filter gives a grayscale image
the same response the RGB filter gave a gray-ish RGB image. Nothing is lost and
nothing is invented.

**`segmentation_models_pytorch` already does this** for `in_channels=1`
(`patch_first_conv` takes the `weight.sum(1, keepdim=True)` branch). This
project does not take that on trust across versions, because the failure is
silent: a model that quietly reinitialised its stem still trains, just worse,
and nothing raises. So `src/model.py` builds a reference three-channel encoder,
sums *its* first-conv weights, and compares. The cell below prints which of the
three things smp actually did, and whether the summed weights had to be written
in — `strategy: sum` means smp was right and nothing was overridden,
`sum-forced` means it was not and we fixed it.

The cell also shows the freeze schedule. The decoder starts random, and its
first gradients are noise; letting that into a pretrained encoder in epoch 1
erases the pretraining before it is ever used. Note that freezing sets
`requires_grad = False` **and** puts the encoder's BatchNorm layers in eval
mode — a "frozen" encoder whose BN is still updating `running_mean` is not
frozen, it is drifting, and nothing about that raises either.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from src import dataset as ds
from src import losses as losses_mod
from src import model as model_mod
from src import paths as paths_mod

FOLD = "dev"          # step 3's alias for fold_uhcs2: same protocol, smallest val
MAGNITUDE = 12.0      # logit magnitude that means "confidently this mask":
                      # sigmoid(-12) = 6e-6, so an "empty" prediction really is
                      # empty rather than 3 pixels of leaked probability mass

ds_settings = ds.load_config()
model_settings = model_mod.load_config()
PATCH = int(ds_settings["patch_size"])

print("model config (configs/default.yaml -> model:)")
for key, value in model_settings.items():
    print(f"  {key:<22} {value!r}")

t0 = time.perf_counter()
net = model_mod.build_model(settings=model_settings)
print(f"\nbuilt in {time.perf_counter() - t0:.1f} s\n")
print(model_mod.describe(net))

report = net.build_report
print(f"\nfirst-conv adaptation detail")
print(f"  smp produced       : {report['smp_did']}")
print(f"  strategy           : {report['strategy']}")
print(f"  overridden by us   : {report['overridden']}")
print(f"  verified           : {report['verified']}")
if "max_abs_diff_from_sum" in report:
    print(f"  |w - sum(w_rgb)|max: {report['max_abs_diff_from_sum']:.3e}")

print("\nfreeze schedule (model.freeze_encoder_epochs = "
      f"{model_settings['freeze_encoder_epochs']})")
n_freeze = int(model_settings["freeze_encoder_epochs"])
for epoch in sorted({0, max(0, n_freeze - 1), n_freeze, n_freeze + 1}):
    state = model_mod.apply_freeze_schedule(net, epoch, model_settings)
    counts = model_mod.summarize(net)
    print(f"  epoch {epoch}: encoder {'FROZEN ' if state['frozen'] else 'trainable'}"
          f"  trainable {counts['trainable']:>11,} / {counts['total']:,}"
          f"  BN layers in eval {model_mod.encoder_state(net)['norm_layers_in_eval']}"
          f"/{state['norm_layers']}"
          f"{'   <- changed here' if state['changed'] else ''}")

## One forward pass, and the proof that the head is not squashing anything

The loss is `BCEWithLogits`-based, which fuses the sigmoid into a numerically
stable expression. If the model applied its own sigmoid, the loss would apply a
second one: predictions would be squeezed into `sigmoid([0, 1]) = [0.5, 0.73]`,
the gradient would be tiny everywhere, and **nothing would raise**. The model
would just train badly and the notebook would show plausible-looking numbers.

So two things are checked here rather than assumed. First the built graph is
walked for any `Sigmoid`, `Softmax` or `Tanh` module — `activation=None` was
passed, but a changed default in a future smp release would not announce
itself. Second the actual output is inspected: logits are unbounded, so a real
logit map straddles zero and reaches well outside `[0, 1]`. If the minimum and
maximum both sat inside `[0, 1]`, something is squashing them.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = net.to(device)
model_mod.unfreeze_encoder(net)     # sizing and demos use the full network

net.eval()
x = torch.randn(2, 1, PATCH, PATCH, device=device)
with torch.no_grad():
    logits = net(x)

print(f"device        : {device}")
print(f"input  shape  : {tuple(x.shape)}")
print(f"output shape  : {tuple(logits.shape)}")
print(f"spatial preserved: {tuple(logits.shape[2:]) == tuple(x.shape[2:])}")
print(f"\nlogit range   : min {logits.min().item():+.4f}   "
      f"max {logits.max().item():+.4f}   mean {logits.mean().item():+.4f}")
print(f"as probabilities after sigmoid: "
      f"{torch.sigmoid(logits).min().item():.4f} .. "
      f"{torch.sigmoid(logits).max().item():.4f}")

forbidden = model_mod.assert_returns_logits(net)   # raises if any were found
print(f"\nactivation modules found in the graph: {forbidden or 'none'}")
logits_unbounded = bool(logits.min().item() < 0.0 < logits.max().item())
print(f"output straddles zero (unbounded, not a probability): {logits_unbounded}")

## The three terms of the loss, and why each one is there

$$L = w_{bce}\,\mathrm{BCE}(\text{pos\_weight}) + w_{dice}\,\mathrm{SoftDice} + w_{cldice}\,\mathrm{clDice}$$

**Weighted BCE — without it the model does not start.** Boundaries are a
minority class: 4.8% of pixels on `fold_uhcs2`'s training split, 15.4% on
`fold_MetalDam`'s validation split. That imbalance is real but it is not
extreme — this is not a 0.1% needle-in-a-haystack problem, and the fix is a
class weight rather than anything exotic. What unweighted BCE does here is
specific and worth being able to recognise: predicting background everywhere
scores 88–95% of pixels correct, so it is a local optimum the optimizer walks
into on the first few batches and leaves only slowly. The model outputs an
empty map, the pixel accuracy looks excellent, and the boundary F1 is zero.
That is exactly why this project never reports pixel accuracy.

`pos_weight = n_negative / n_positive` on the fold's own training split. It is
read from `configs/fold_stats.yaml` for the fold being trained and there is no
default: the folds differ by a factor of 2.2, so one constant would mis-weight
two of the three.

**Dice — scale-free overlap.** BCE is a mean over pixels, so a tile with few
boundary pixels contributes a small gradient; Dice measures overlap as a
*fraction of what is there*, so a sparse tile pushes as hard as a dense one.
Dice is also what makes the loss care about the shape of the prediction rather
than its per-pixel calibration.

**clDice — connectivity, which Dice cannot see.** Dice counts pixels. Break a
boundary in twenty places and you lose maybe 3% of the overlap: Dice calls that
prediction 97% right. Downstream it is not 97% right. This prior feeds a
watershed in Phase 1, and a broken boundary is not a slightly worse boundary —
it is a hole through which two grains merge into one region. clDice compares
each mask against the other's *soft skeleton*, so a gap costs it topology
recall, and thickening a line — which Dice punishes hard — costs it almost
nothing.

Everything is computed from raw logits. BCE uses the fused
`binary_cross_entropy_with_logits`; Dice and clDice apply their own sigmoid
internally. The logits are deliberately **not** sigmoided once and passed to
all three, because that would force BCE onto the unfused path where `log(0)`
is reachable.

In [ ]:
loss_settings = losses_mod.load_config()
fold_stats = ds.load_fold_stats()

print("loss config (configs/default.yaml -> loss:)")
for key, value in loss_settings.items():
    print(f"  {key:<16} {value!r}")

criterion = losses_mod.BoundaryLoss.for_fold(
    FOLD, settings=loss_settings, fold_stats=fold_stats).to(device)
print(f"\n{criterion.describe()}")
print(f"  pos_weight for '{FOLD}' came from configs/fold_stats.yaml: "
      f"{float(criterion.pos_weight):.3f} = n_negative / n_positive on its "
      f"train split")

## How large a batch actually fits — with the optimizer state, not without it

The number people usually quote is the forward pass, and it is optimistic by
roughly a factor of three. What a training step actually holds at once is:

1. the parameters (~24 M for a ResNet-34 U-Net, ~93 MB in fp32),
2. every intermediate activation from the forward pass, kept alive because the
   backward pass needs them — this is the term that scales with batch size,
3. the gradients, one per parameter,
4. **the optimizer state**: Adam keeps `exp_avg` and `exp_avg_sq`, so another
   two floats per parameter — but they are allocated lazily, on the *first*
   `step()`. Time only the forward pass and you never see them, and the batch
   size you pick then OOMs in the first epoch.

So each trial below runs a real forward, a real backward and two real optimizer
steps, and reports `torch.cuda.max_memory_allocated()` afterwards. The cache is
emptied and the peak counter reset between trials, or each measurement would
inherit the previous one's high-water mark.

The loss used is the real compound loss, not a placeholder: clDice's
soft-skeletonize is three rounds of erode-and-open, each of which allocates a
tensor the size of the prediction, and leaving it out would under-count.

The encoder is unfrozen for this measurement. Freezing it removes its gradients
and its optimizer state, so a batch size measured under a frozen encoder is one
that OOMs at epoch `freeze_encoder_epochs` — the worst case is what has to fit.

**The rule for choosing:** the largest batch that fits *and* leaves at least
15% of the card free. A batch that fits with 200 MB to spare will not survive
a fragmented allocator, a slightly larger tile, or a second process on the same
GPU.

These are real optimizer steps on random noise, so afterwards the weights are
no longer the pretrained ones. The cell rebuilds the model from config at the
end rather than letting anything downstream inherit twelve steps of gradient
descent on white noise.

In [ ]:
import gc

BATCH_SIZES = [8, 16, 32, 64]
HEADROOM = 0.15          # fraction of total VRAM to leave free
total_vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
                 if device.type == "cuda" else float("nan"))


def _reset():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def size_trial(batch_size, warmup=2):
    """One honest training step: forward, backward, optimizer step, peak memory."""
    _reset()
    net.zero_grad(set_to_none=True)
    optimizer = torch.optim.AdamW(net.parameters(), lr=1e-4)
    record = {"batch": batch_size, "fits": False, "peak_gb": float("nan"),
              "seconds": float("nan"), "tiles_per_second": float("nan"),
              "note": ""}
    try:
        images = torch.randn(batch_size, 1, PATCH, PATCH, device=device)
        masks = (torch.rand(batch_size, 1, PATCH, PATCH, device=device)
                 < 0.10).float()
        # Two full steps first: Adam allocates exp_avg / exp_avg_sq on step 1,
        # so a peak measured before that is missing two floats per parameter.
        for _ in range(warmup):
            optimizer.zero_grad(set_to_none=True)
            criterion(net(images), masks).backward()
            optimizer.step()
        if device.type == "cuda":
            torch.cuda.synchronize()
        started = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        criterion(net(images), masks).backward()
        optimizer.step()
        if device.type == "cuda":
            torch.cuda.synchronize()
        record["seconds"] = time.perf_counter() - started
        record["tiles_per_second"] = batch_size / record["seconds"]
        record["peak_gb"] = (torch.cuda.max_memory_allocated() / 1024 ** 3
                             if device.type == "cuda" else float("nan"))
        record["fits"] = True
    except RuntimeError as exc:
        # torch.cuda.OutOfMemoryError subclasses RuntimeError, and older
        # versions raise a plain RuntimeError. Anything that is NOT an OOM is
        # a real bug and must not be swallowed as "does not fit".
        if "out of memory" not in str(exc).lower() \
                and type(exc).__name__ != "OutOfMemoryError":
            raise
        record["note"] = "CUDA OOM"
    finally:
        # Drop every reference this trial holds before measuring the next one.
        # Plain rebinding, not del: an OOM can strike before `images` exists.
        images = masks = optimizer = None
        net.zero_grad(set_to_none=True)
        _reset()
    return record


net.train()
model_mod.unfreeze_encoder(net)
if device.type != "cuda":
    print("! no GPU in this session: memory figures are unavailable and the "
          "timings are CPU timings. The batch size written below is the "
          "largest that ran, not the largest that fits a T4.")

trials = []
for batch_size in BATCH_SIZES:
    record = size_trial(batch_size)
    trials.append(record)
    print(f"  batch {batch_size:>3}: "
          + (f"peak {record['peak_gb']:5.2f} GB  "
             f"{record['seconds'] * 1000:7.1f} ms/step  "
             f"{record['tiles_per_second']:6.1f} tiles/s"
             if record["fits"] else f"{record['note']}"))

fitting = [r for r in trials if r["fits"]]
if not fitting:
    raise RuntimeError(
        "no batch size completed a training step; the model cannot be trained "
        "on this host at patch 256.")

if device.type == "cuda":
    roomy = [r for r in fitting if r["peak_gb"] <= (1 - HEADROOM) * total_vram_gb]
    chosen = max(roomy or fitting, key=lambda r: r["batch"])
    rule = ("largest that fits with >= 15% of the card free"
            if roomy else "largest that fits (nothing left 15% free)")
else:
    chosen = max(fitting, key=lambda r: r["batch"])
    rule = "largest that ran (CPU session, memory not measured)"

table = pd.DataFrame(trials)
table["fits"] = table["fits"].map({True: "yes", False: "OOM"})
table["chosen"] = ["  <-- chosen" if r["batch"] == chosen["batch"] else ""
                   for r in trials]
if device.type == "cuda":
    table["% of card"] = (table["peak_gb"] / total_vram_gb * 100).round(1)
print(f"\nGPU: {PATHS['gpu']}  ({total_vram_gb:.1f} GB total)"
      if device.type == "cuda" else "\nCPU session")
print(table.to_string(index=False))
print(f"\nrule: {rule}  ->  batch_size = {chosen['batch']}")

written = paths_mod.set_config_scalar(
    "train", "batch_size", int(chosen["batch"]),
    note=(f"measured on {PATHS['platform']} / {PATHS['gpu']}, patch {PATCH}, "
          f"peak {chosen['peak_gb']:.2f} GB incl. AdamW state"))
print(f"\nconfigs/default.yaml line {written['line']}:")
print(f"    was: {written['before']}")
print(f"    now: {written['after']}")

# The sweep ran real optimizer steps on random noise, so these weights are no
# longer the pretrained ones. Nothing downstream should inherit that, so the
# model is rebuilt from config -- which is also what step 6 will do.
net = model_mod.build_model(settings=model_settings).to(device)
print(f"\nmodel rebuilt from config after the sweep "
      f"({net.build_report['strategy']}); the sizing steps did not leave "
      f"their noise in it")

## The loss on real boundaries: four predictions, three terms

Synthetic grids prove the arithmetic; they do not prove anything about
micrographs, whose boundaries curve, meet at triple points and vary in density
by a factor of thirty across the fold. So the demo runs on two real tiles from
the `dev` fold — the densest MetalDam tile and the sparsest Steel1 tile — and
scores four constructed predictions against each tile's own ground truth:

| prediction | what it is | what it should cost |
| --- | --- | --- |
| **truth** | the ground truth itself | nothing, or the terms are broken |
| **gap** | the line severed periodically | the error that matters downstream |
| **dilated** | the line one pixel fatter all round | many wrong pixels, topology intact |
| **empty** | no boundary anywhere | the local optimum weighted BCE exists to escape |

Every prediction is turned into logits (`±10`) and passed through the *real*
criterion, not a reimplementation of it — the numbers below are produced by the
code that will train the model.

One detail about the gap. The ground truth is dilated to
`boundary_gt.line_width_px = 2`, and erasing a single pixel from a two-pixel
line leaves the other row intact: the line is still connected and nothing has
been demonstrated. So a gap is one pixel *long* along the line and one
line-width *across* it — a 3×3 block. `cut_gaps` spaces those blocks every 180
boundary pixels, which removes about the same *fraction* of the boundary on the
dense tile and the sparse one, so the two are comparable.

The tiles come from the persistent cache `04_dataset.ipynb` wrote, if it is
there. The whole fold is not preloaded: a cold preload is ~1,000 Drive reads at
~0.6 s each and this cell needs two images, so it asks the cache for those two
and falls back to reading them directly.

In [ ]:
reports_dir = Path(PATHS["reports_dir"])
manifest = reports_dir / ds_settings["manifest_subdir"] / f"{FOLD}.csv"
train_rows = ds.load_manifest(manifest, "train")
crops = ds.load_crops(reports_dir)
roots = {"data_root": Path(PATHS["data_root"]),
         "gt_root": Path(PATHS["gt_boundaries_root"])}


def pick(dataset_name, densest):
    candidates = sorted([r for r in train_rows if r["dataset"] == dataset_name],
                        key=lambda r: r["boundary_fraction"], reverse=densest)
    if not candidates:
        raise RuntimeError(
            f"no {dataset_name} tiles in {manifest.name}; the fold protocol "
            "changed and this cell's choice of tiles needs revisiting.")
    return candidates[0]


dense_row = pick("MetalDam", densest=True)
sparse_row = pick("Steel1", densest=False)
needed = {(r["dataset"], r["source_image"]) for r in (dense_row, sparse_row)}

cache_dir = Path(PATHS["persistent_dir"]) / ds_settings["cache_subdir"]
warm, cache_reason = ds.read_persistent_cache(
    cache_dir, f"{FOLD}-train", ds.cache_key(train_rows), required=needed)
print(f"tile cache: {'WARM' if warm else 'not used'} -- {cache_reason}")


def tile_of(row):
    if warm is not None:
        image_full, gt_full = warm[(row["dataset"], row["source_image"])]
    else:
        image_full, gt_full = ds.read_pair(row, crops=crops, roots=roots)
    patch = int(row["patch"])
    image = ds.crop_tile(image_full, row["x"], row["y"], patch)
    mask = (ds.crop_tile(gt_full, row["x"], row["y"], patch) > 0).astype(np.uint8)
    return image, mask


GAP_SPACING, GAP_WIDTH = 180, 3
ORDER_CASES = ["truth", "gap", "dilated", "empty"]
tiles, records = {}, []
for label, row in (("MetalDam (dense)", dense_row), ("Steel1 (sparse)", sparse_row)):
    image, truth = tile_of(row)
    variants = {
        "truth": truth,
        "gap": losses_mod.cut_gaps(truth, spacing=GAP_SPACING, gap=GAP_WIDTH),
        "dilated": losses_mod.dilate_mask(truth, iterations=1),
        "empty": np.zeros_like(truth),
    }
    tiles[label] = {"row": row, "image": image, "variants": variants}
    target = losses_mod.as_target(truth).to(device)
    print(f"\n{label}  {row['tile_id']}")
    print(f"  boundary fraction {truth.mean():.4f} ({int(truth.sum())} px); "
          f"gap removes {1 - variants['gap'].sum() / max(1, truth.sum()):.2%}, "
          f"dilation adds {variants['dilated'].sum() / max(1, truth.sum()) - 1:.2%}")
    for name, variant in variants.items():
        with torch.no_grad():
            terms = criterion.components(
                losses_mod.as_logits(variant, MAGNITUDE).to(device), target)
        records.append({"tile": label, "prediction": name,
                        "bce": float(terms["bce"]),
                        "dice": float(terms["dice"]),
                        "cldice": float(terms["cldice"]),
                        "total": float(terms["total"])})

demo = pd.DataFrame(records)
print("\n" + demo.to_string(index=False,
                            float_format=lambda v: f"{v:10.5f}"))

# The comparison the clDice term exists for, pulled out of the table.
ratios = {}
for label in tiles:
    sub = demo[demo["tile"] == label].set_index("prediction")
    ratios[label] = {
        "dice_gap": sub.loc["gap", "dice"], "cldice_gap": sub.loc["gap", "cldice"],
        "dice_dilated": sub.loc["dilated", "dice"],
        "cldice_dilated": sub.loc["dilated", "cldice"],
    }
    r = ratios[label]
    print(f"\n{label}: Dice charges {r['dice_dilated'] / max(1e-9, r['dice_gap']):.1f}x "
          f"MORE for fattening the line than for cutting it; clDice charges "
          f"{r['cldice_dilated'] / max(1e-9, r['cldice_gap']):.2f}x.")

## What the soft skeleton actually does to a 2-px line — measured, not assumed

The `dilated` column of the table above is the interesting one, and reading it
requires knowing what `soft_skeletonize` produces on boundaries this thin. That
was got wrong once in this project by reasoning about it instead of looking, so
this cell looks.

Two synthetic probes first, because their answers are unambiguous: an isolated
2-px line, and a 2-px grid whose lines cross. Then the real tiles, through
`losses.cldice_parts`, which returns the intermediate quantities the clDice term
is built from rather than a re-implementation of it sitting next to it.

The question the real tiles have to answer is specific. A clDice loss of
exactly `0.0000` has two completely different explanations and the loss value
alone cannot tell them apart:

- **topology invariance** — both skeletons are real, they lie on top of each
  other, and the term is correctly saying the centreline did not move;
- **degeneracy** — the predicted skeleton is *empty*, so `t_prec` is
  `smooth / smooth = 1` by construction and the term is saying nothing at all.

So the skeleton pixel counts are printed for every case. If
`skel_pred_sum` is 0, the number below it is meaningless.

In [ ]:
ITERS = int(loss_settings["cldice_iters"])


def skeleton_of(mask):
    """The soft skeleton of a binary mask, as numpy, via the real function."""
    tensor = losses_mod.as_target(mask)
    skel = losses_mod.soft_skeletonize(tensor, ITERS)
    return tensor[0, 0].numpy(), skel[0, 0].numpy()


def show(block, title):
    print(f"  {title}")
    for row in block:
        print("    " + "".join("#" if v > 0.5 else ("+" if v > 1e-6 else ".")
                               for v in row))


probe_line = np.zeros((48, 48), dtype=np.uint8)
probe_line[22:24, 6:42] = 1
probe_grid = np.zeros((48, 48), dtype=np.uint8)
for k in (10, 26, 42):
    probe_grid[k:k + 2, :] = 1
    probe_grid[:, k:k + 2] = 1

probes = {}
for name, probe in (("isolated 2 px line", probe_line),
                    ("2 px grid with junctions", probe_grid)):
    inp, skel = skeleton_of(probe)
    diff = np.abs(inp - skel)
    probes[name] = {"input_px": float(inp.sum()), "skeleton_px": float(skel.sum()),
                    "max_abs_diff": float(diff.max()),
                    "differing_px": int((diff > 1e-6).sum()),
                    "input": inp, "skeleton": skel}
    print(f"{name} (cldice_iters={ITERS})")
    print(f"  input {probes[name]['input_px']:.0f} px, "
          f"skeleton {probes[name]['skeleton_px']:.0f} px, "
          f"max |input - skeleton| {probes[name]['max_abs_diff']:.4f}, "
          f"{probes[name]['differing_px']} px differ")

# The grid, around one junction, printed rather than described.
window = (slice(6, 16), slice(6, 16))
print("\naround the junction at (10, 10) of the grid probe:")
show(probes["2 px grid with junctions"]["input"][window], "input")
show(probes["2 px grid with junctions"]["skeleton"][window], "soft skeleton")
print("  ('#' = 1, '+' = between 0 and 1, '.' = 0)")

# Now the real tiles, case by case.
skel_parts = {}
for label, info in tiles.items():
    target = losses_mod.as_target(info["variants"]["truth"]).to(device)
    skel_parts[label] = {}
    print(f"\n{label}")
    print(f"  {'prediction':<10} {'pred px':>9} {'true px':>9} "
          f"{'skel(pred)':>11} {'skel(true)':>11} {'skel(p) on t':>13} "
          f"{'t_prec':>8} {'t_rec':>8} {'clDice loss':>12}")
    for name in ORDER_CASES:
        variant = info["variants"][name]
        parts = losses_mod.cldice_parts(
            losses_mod.as_logits(variant, MAGNITUDE).to(device), target,
            iters=criterion.cldice_iters, smooth=criterion.smooth,
            eps=criterion.eps)
        skel_parts[label][name] = parts
        print(f"  {name:<10} {parts['pred_sum']:>9.0f} {parts['true_sum']:>9.0f} "
              f"{parts['skel_pred_sum']:>11.0f} {parts['skel_true_sum']:>11.0f} "
              f"{parts['skel_pred_on_true']:>13.0f} "
              f"{parts['t_prec']:>8.5f} {parts['t_rec']:>8.5f} "
              f"{parts['loss']:>12.5f}")

for label, cases in skel_parts.items():
    dil = cases["dilated"]
    verdict = ("DEGENERATE: the predicted skeleton is empty, so t_prec = "
               "smooth/smooth = 1 and the low clDice means nothing"
               if dil["skel_pred_sum"] < 0.25 * dil["true_sum"] else
               "GENUINE topology invariance: the predicted skeleton is real, "
               "non-empty and sits on the true boundary")
    print(f"\n{label}, dilated case -> {verdict}")
    print(f"    skeleton of the dilated prediction: {dil['skel_pred_sum']:.0f} px, "
          f"of which {dil['skel_pred_on_true']:.0f} px "
          f"({dil['skel_pred_on_true'] / max(1e-9, dil['skel_pred_sum']):.1%}) "
          f"lie on the true boundary; the truth itself is "
          f"{dil['true_sum']:.0f} px and its own skeleton "
          f"{dil['skel_true_sum']:.0f} px")

## The same numbers as a picture

Log scale on the vertical axis, floored at `1e-4`: the terms span five orders
of magnitude between a perfect prediction and an empty one, and a linear axis
would show four bars and three invisible ones. The left column is the tile with
its ground truth in red, so the density difference between the two rows is
visible rather than asserted.

In [ ]:
import matplotlib.pyplot as plt

TERMS = ["bce", "dice", "cldice"]
ORDER = ORDER_CASES
FLOOR = 1e-4

fig, axes = plt.subplots(len(tiles), 2, figsize=(16, 5.5 * len(tiles)),
                         gridspec_kw={"width_ratios": [1, 1.6]})
for ax_row, (label, info) in zip(np.atleast_2d(axes), tiles.items()):
    image, truth = info["image"], info["variants"]["truth"]
    shown = (image - image.min()) / max(1e-6, float(np.ptp(image)))
    rgb = np.dstack([shown] * 3)
    rgb[..., 0] = np.where(truth > 0, 1.0, rgb[..., 0])
    rgb[..., 1] = np.where(truth > 0, rgb[..., 1] * 0.2, rgb[..., 1])
    rgb[..., 2] = np.where(truth > 0, rgb[..., 2] * 0.2, rgb[..., 2])
    ax_row[0].imshow(rgb)
    ax_row[0].set_title(f"{label}\n{info['row']['tile_id']}  "
                        f"boundary {truth.mean():.3f}", fontsize=10)
    ax_row[0].axis("off")

    sub = demo[demo["tile"] == label].set_index("prediction")
    width, positions = 0.26, np.arange(len(ORDER))
    for i, term in enumerate(TERMS):
        values = [max(FLOOR, sub.loc[case, term]) for case in ORDER]
        bars = ax_row[1].bar(positions + (i - 1) * width, values, width,
                             label=term)
        for rect, raw in zip(bars, [sub.loc[case, term] for case in ORDER]):
            ax_row[1].text(rect.get_x() + rect.get_width() / 2,
                           rect.get_height() * 1.15, f"{raw:.3f}",
                           ha="center", fontsize=7, rotation=90)
    ax_row[1].set_yscale("log")
    ax_row[1].set_ylim(FLOOR, 100)
    ax_row[1].set_xticks(positions)
    ax_row[1].set_xticklabels(ORDER)
    ax_row[1].set_ylabel("loss term (log scale)")
    ax_row[1].set_title(f"{label}: what each term charges for each error",
                        fontsize=10)
    ax_row[1].legend(loc="upper left")
    ax_row[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## What to read off that chart

**The `empty` column is why `pos_weight` exists.** BCE charges it something
like 10, three orders of magnitude above anything else on the chart. Remove the
class weight and that column drops by exactly a factor of `pos_weight` — the
checks cell measures that — at which point predicting nothing is competitive
with predicting something for the first few epochs, and since it is also the
easiest thing to predict, that is where the optimizer goes. Dice and clDice
both saturate near 1.0 there and stop discriminating; BCE is the term that
pushes the model off that optimum.

**The `gap` column is the error that matters, and both overlap terms
under-charge it.** Cutting the boundary in dozens of places removes ~3% of the
pixels, so Dice scores it ~0.97 — nearly perfect. It is not nearly perfect.
This prior feeds a watershed in Phase 1, and each gap is a channel through
which two grains merge into one region. One broken boundary is a wrong
segmentation, not a slightly noisy one.

**The `dilated` column is where the two overlap terms part company, and it is
the reason clDice is in the loss.** One pixel of extra width roughly doubles
the predicted boundary pixels. Dice charges 0.335 and 0.390 for that on the two
tiles — an order of magnitude more than it charges for the severed line. clDice
charges 0.007 and 0.000. A dilated line has the same centreline, and that is
all clDice is looking at.

### The correction: clDice removes Dice's thickness bias, it does not add gap sensitivity

An earlier version of this notebook claimed clDice penalises a gap *more* than
Dice does, on the reasoning that a 2-px line is destroyed by one erosion, so
`skeleton(g) == g`. **The measurement cell above shows that is only half
true, and the half it gets wrong is the half the claim rested on.**

- On an **isolated** 2-px line the skeleton *is* the line: every line pixel's
  3×1 min-pool reaches a background row, so `open(x)` is empty and
  `relu(x - open(x)) = x`. That much was right.
- At a **junction** it is not. Where two 2-px lines cross, the shape is three
  or more pixels thick in *both* directions, erosion survives on the crossing,
  the opening dilates that core back out, and `relu(x - open(x))` carves a hole
  out of the skeleton at every junction. Real boundary networks are made of
  junctions. The printed grid probe shows the hole.

So the skeleton of the ground truth here is the mask minus a patch at each
junction — neither the mask itself nor a thinned centreline — and the tidy
argument that clDice's gap penalty must therefore exceed Dice's does not
follow from anything. The numbers do not support it either.

What the numbers *do* support, and what the tests now assert:

| | Dice | clDice |
| --- | --- | --- |
| severed line (~3% of pixels wrong) | 0.017 / 0.021 | comparable |
| line one pixel fatter (~100% more pixels) | **0.335 / 0.390** | **0.007 / 0.000** |

clDice is nearly blind to thickness and Dice is dominated by it. That is worth
having for a reason specific to this ground truth: **thickness here is a
convention, not a measurement.** Step 2 skeletonized every boundary and dilated
it back to a uniform `boundary_gt.line_width_px = 2`. The 2 is a choice. A loss
governed by Dice alone spends most of its gradient making the predicted band
exactly that arbitrary width; adding clDice puts weight back on *where the line
runs* instead. Stated plainly: **clDice's contribution here is removing Dice's
thickness bias, not adding gap sensitivity.**

Two consequences worth carrying into step 6:

- Nothing in this loss charges a break much more than it charges a few percent
  of misplaced pixels. If the trained model produces broken boundaries, the
  fix is not turning `w_cldice` up — on this evidence that would mostly buy
  more thickness tolerance. The lever with actual leverage is a thicker
  `boundary_gt.line_width_px`, which would give the soft skeleton a genuine
  centreline to compare, or an explicit connectivity metric at evaluation time.
- Report boundary F1 and clDice separately per dataset. A model that scores
  well on Dice and badly on clDice is painting bands in roughly the right
  places; the opposite means thin, correctly-routed, badly-calibrated lines.
  Those need different fixes.

## `pos_weight` across the four folds

The four numbers below are not four attempts at the same quantity. Each is
`n_negative / n_positive` on a *different* training set, because leave-one-
dataset-out changes what is being trained on.

`fold_MetalDam` is the one to look at. It holds MetalDam out, so it trains on
Steel1, uhcs1 and uhcs2 — mean boundary fraction 0.059 — and validates on
MetalDam plus Steel1's validation parents, mean boundary fraction 0.154. It
trains on sparse boundaries and is scored on dense ones. That is why its
`pos_weight` is 15.974 against roughly 7.1 for the other folds, a factor of
2.2.

This is a property of the design, not a bug: MetalDam genuinely is denser than
everything else, and a fold that holds it out genuinely has less boundary to
learn from. Two consequences follow and both matter in step 6:

- **`fold_MetalDam`'s loss values are not comparable to the other folds'.** A
  BCE term scaled by 15.974 and one scaled by 7.111 are different units. Compare
  folds on boundary F1 and on clDice, never on the loss.
- **It is the hardest fold**, and a drop in its score is expected rather than
  alarming. It is also the most informative one, because it is the largest
  domain shift in the set.

In [ ]:
rows = []
for name in ("fold_MetalDam", "fold_uhcs1", "fold_uhcs2", "dev"):
    entry = fold_stats["folds"][name]
    rows.append({
        "fold": name,
        "held out": entry["held_out"],
        "alias of": entry.get("alias_of") or "-",
        "pos_weight": losses_mod.fold_pos_weight(name, fold_stats=fold_stats),
        "train frac": entry["train_boundary_fraction"]["mean"],
        "val frac": entry["val_boundary_fraction"]["mean"],
        "train tiles": entry["n_train_tiles"],
        "val tiles": entry["n_val_tiles"],
    })
folds_table = pd.DataFrame(rows)
folds_table["val/train density"] = (folds_table["val frac"]
                                    / folds_table["train frac"]).round(2)
print(folds_table.to_string(index=False))

weights = dict(zip(folds_table["fold"], folds_table["pos_weight"]))
spread = max(weights.values()) / min(weights.values())
print(f"\nspread across folds: {spread:.2f}x "
      f"({min(weights.values()):.3f} .. {max(weights.values()):.3f})")
print("A single constant pos_weight would be wrong by that factor on two of "
      "the three folds, which is why src/losses.py has no default and raises "
      "on an unknown fold name.")
try:
    losses_mod.fold_pos_weight("fold_not_a_real_fold", fold_stats=fold_stats)
except losses_mod.LossError as exc:
    print(f"\nunknown fold -> {type(exc).__name__}: {str(exc).splitlines()[0]}")

## Run the test suite

The tests are where this step is actually proved, and they run here because
there is no local environment to run them in. They cover the things that fail
*silently*: a term that returns NaN on an empty tile, a loss that is not
monotone on the path to the truth, a `pos_weight` quietly defaulting instead of
raising, a gradient that does not reach the model input.

`test_dataset.py` is run too. `src/losses.py` reads `fold_stats.yaml` through
`src/dataset.py`, so a change here that broke step 4 would otherwise show up
two notebooks later.

In [ ]:
import subprocess

pytest_run = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "--tb=short", "-p", "no:cacheprovider",
     str(Path(PATHS["repo_root"]) / "tests")],
    cwd=str(PATHS["repo_root"]), capture_output=True, text=True)

print(pytest_run.stdout[-8000:])
if pytest_run.stderr.strip():
    print("stderr:", pytest_run.stderr[-2000:])
print(f"pytest exit code: {pytest_run.returncode}")

## Checks

Where the step is declared correct or not. These re-verify on the real fold
what the unit tests verify on constructed data — a passing unit test and a
working model are different claims.

In [ ]:
checks = []


def check(name, ok, detail=""):
    checks.append((name, bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}  {name}{'  -- ' + detail if detail else ''}")


# -- model ---------------------------------------------------------------
net.eval()
probe = torch.randn(2, 1, PATCH, PATCH, device=device)
with torch.no_grad():
    probe_logits = net(probe)
check("output spatial shape equals input",
      tuple(probe_logits.shape[2:]) == tuple(probe.shape[2:])
      and probe_logits.shape[0] == probe.shape[0] and probe_logits.shape[1] == 1,
      f"{tuple(probe.shape)} -> {tuple(probe_logits.shape)}")
check("no activation module in the graph (head returns logits)",
      not model_mod.assert_returns_logits(net), "walked every named module")
check("logits are unbounded, not probabilities",
      bool(probe_logits.min().item() < 0.0 < probe_logits.max().item()),
      f"min {probe_logits.min().item():+.3f}, max {probe_logits.max().item():+.3f}")
check("first conv adapted by summing the pretrained RGB filters",
      net.build_report["strategy"] in ("sum", "sum-forced")
      and net.build_report["verified"],
      f"strategy={net.build_report['strategy']}, "
      f"smp did {net.build_report['smp_did']!r}, "
      f"overridden={net.build_report['overridden']}")
check("first conv takes exactly one input channel",
      model_mod.first_conv(net.encoder).weight.shape[1] == 1,
      f"{tuple(model_mod.first_conv(net.encoder).weight.shape)}")

frozen = model_mod.apply_freeze_schedule(net, 0, model_settings)
frozen_state = model_mod.encoder_state(net)
thawed = model_mod.apply_freeze_schedule(
    net, int(model_settings["freeze_encoder_epochs"]), model_settings)
thawed_state = model_mod.encoder_state(net)
check("encoder freezes at epoch 0 and unfreezes at epoch N",
      frozen_state["fully_frozen"] and not thawed_state["fully_frozen"],
      f"epoch 0: {frozen_state['encoder_frozen_params']:,} params frozen, "
      f"{frozen_state['norm_layers_in_eval']}/{frozen_state['norm_layers']} BN "
      f"in eval; epoch {model_settings['freeze_encoder_epochs']}: "
      f"{thawed_state['encoder_trainable_params']:,} params trainable")

# -- loss on real tiles ---------------------------------------------------
finite = demo[["bce", "dice", "cldice", "total"]]
check("every loss term finite on real tiles",
      bool(np.isfinite(finite.to_numpy()).all()),
      f"{finite.size} values over {len(tiles)} tiles x {len(ORDER)} predictions")
check("every loss term non-negative on real tiles",
      bool((finite.to_numpy() >= 0).all()),
      f"min {finite.to_numpy().min():.3e}")
perfect = demo[demo["prediction"] == "truth"][["bce", "dice", "cldice", "total"]]
check("the ground truth itself scores ~0 on every term and both tiles",
      bool((perfect.to_numpy() < 1e-2).all()),
      f"largest term on a perfect prediction {perfect.to_numpy().max():.3e}")

# -- what the soft skeleton actually does (measured above, asserted here) --
line_probe = probes["isolated 2 px line"]
grid_probe = probes["2 px grid with junctions"]
check("soft skeleton of an ISOLATED 2 px line is the line itself",
      line_probe["max_abs_diff"] < 1e-6,
      f"input {line_probe['input_px']:.0f} px, skeleton "
      f"{line_probe['skeleton_px']:.0f} px, max |diff| "
      f"{line_probe['max_abs_diff']:.2e}")
check("soft skeleton CARVES JUNCTIONS out of a 2 px grid",
      grid_probe["differing_px"] > 0
      and grid_probe["differing_px"] < 0.25 * grid_probe["input_px"],
      f"input {grid_probe['input_px']:.0f} px, skeleton "
      f"{grid_probe['skeleton_px']:.0f} px, {grid_probe['differing_px']} px "
      f"differ ({grid_probe['differing_px'] / grid_probe['input_px']:.1%}) -- "
      "so skeleton(g) != g on a real boundary network")

# -- the property clDice is actually in the loss for ----------------------
check("Dice is dominated by the thickness error",
      all(r["dice_dilated"] > 3.0 * r["dice_gap"] for r in ratios.values()),
      "; ".join(f"{label}: charges {r['dice_dilated'] / max(1e-9, r['dice_gap']):.1f}x "
                f"more for fattening than for cutting"
                for label, r in ratios.items()))
check("clDice is far less sensitive to line thickness than Dice",
      all(r["cldice_dilated"] < 0.25 * r["dice_dilated"]
          for r in ratios.values()),
      "; ".join(f"{label}: dilated dice={r['dice_dilated']:.4f} vs "
                f"cldice={r['cldice_dilated']:.4f}"
                for label, r in ratios.items()))
# A clDice of 0 with an EMPTY predicted skeleton would be t_prec = smooth/smooth
# = 1, which is degeneracy wearing invariance's clothes. Both are checked.
invariance_ok, invariance_detail = True, []
for label, cases in skel_parts.items():
    dil = cases["dilated"]
    real_skeletons = (dil["skel_pred_sum"] > 0.25 * dil["true_sum"]
                      and dil["skel_true_sum"] > 0.25 * dil["true_sum"])
    on_boundary = (dil["skel_pred_on_true"]
                   > 0.8 * max(1e-9, dil["skel_pred_sum"]))
    invariance_ok &= bool(real_skeletons and on_boundary)
    invariance_detail.append(
        f"{label}: skel(pred)={dil['skel_pred_sum']:.0f} px, "
        f"skel(true)={dil['skel_true_sum']:.0f} px, "
        f"{dil['skel_pred_on_true'] / max(1e-9, dil['skel_pred_sum']):.0%} of the "
        f"predicted skeleton on the true boundary (truth {dil['true_sum']:.0f} px)")
check("that insensitivity is topology invariance, not an empty skeleton",
      invariance_ok, "; ".join(invariance_detail))
check("clDice still charges for a severed line, and ranks it above thickness "
      "more strongly than Dice does",
      all(r["cldice_gap"] > 0.0
          and r["cldice_gap"] * r["dice_dilated"]
          > 3.0 * r["cldice_dilated"] * r["dice_gap"]
          for r in ratios.values()),
      "; ".join(f"{label}: gap cldice={r['cldice_gap']:.4f} vs dilated "
                f"cldice={r['cldice_dilated']:.4f}"
                for label, r in ratios.items()))
# What pos_weight actually buys, measured: the cost of predicting nothing,
# with and without it. Dice and clDice both saturate near 1 on an empty
# prediction and cannot tell a sparse tile from a dense one; BCE can, and the
# class weight is the whole of the difference.
plain_bce = losses_mod.BoundaryLoss(1.0, settings=loss_settings).to(device)
empty_ratio = {}
for label, info in tiles.items():
    target = losses_mod.as_target(info["variants"]["truth"]).to(device)
    zeros = losses_mod.as_logits(info["variants"]["empty"], MAGNITUDE).to(device)
    with torch.no_grad():
        weighted = float(criterion.components(zeros, target)["bce"])
        plain = float(plain_bce.components(zeros, target)["bce"])
    empty_ratio[label] = weighted / max(1e-12, plain)
check("pos_weight scales the cost of predicting nothing by exactly pos_weight",
      all(abs(r - float(criterion.pos_weight)) / float(criterion.pos_weight) < 0.05
          for r in empty_ratio.values()),
      "; ".join(f"{label}: {r:.3f}x vs pos_weight "
                f"{float(criterion.pos_weight):.3f}"
                for label, r in empty_ratio.items()))
empty = demo[demo["prediction"] == "empty"]
check("Dice and clDice saturate on an empty prediction, so BCE is what moves it",
      bool((empty["dice"] > 0.9).all() and (empty["cldice"] > 0.9).all()),
      f"dice {empty['dice'].min():.3f}-{empty['dice'].max():.3f}, "
      f"cldice {empty['cldice'].min():.3f}-{empty['cldice'].max():.3f}, "
      f"bce {empty['bce'].min():.3f}-{empty['bce'].max():.3f}")

# -- pos_weight -----------------------------------------------------------
from_file = {name: float(fold_stats["folds"][name]["pos_weight"])
             for name in ("fold_MetalDam", "fold_uhcs1", "fold_uhcs2")}
from_code = {name: losses_mod.fold_pos_weight(name, fold_stats=fold_stats)
             for name in from_file}
check("pos_weight matches configs/fold_stats.yaml for every fold",
      from_code == from_file, f"{from_code}")
check("pos_weight differs between folds by more than 2x",
      max(from_code.values()) / min(from_code.values()) > 2.0,
      f"{max(from_code.values()) / min(from_code.values()):.2f}x "
      f"({min(from_code.values()):.3f} .. {max(from_code.values()):.3f})")
check("the criterion carries the fold's own pos_weight",
      abs(float(criterion.pos_weight) - from_file["fold_uhcs2"]) < 1e-6,
      f"'{FOLD}' -> {float(criterion.pos_weight):.3f} "
      f"(= fold_uhcs2, of which dev is the alias)")
raised = False
try:
    losses_mod.fold_pos_weight("fold_not_a_real_fold", fold_stats=fold_stats)
except losses_mod.LossError:
    raised = True
check("an unknown fold raises instead of falling back", raised)

# -- config ---------------------------------------------------------------
reloaded = paths_mod.load_config()
check("chosen batch size written to configs/default.yaml",
      int((reloaded.get("train") or {}).get("batch_size") or 0) == int(chosen["batch"]),
      f"train.batch_size = {(reloaded.get('train') or {}).get('batch_size')} "
      f"(chosen {chosen['batch']}, {rule})")
check("the model and loss sections survived that edit intact",
      (reloaded.get("model") or {}).get("encoder") == model_settings["encoder"]
      and float((reloaded.get("loss") or {}).get("w_cldice")) == criterion.w_cldice,
      f"model.encoder={reloaded['model']['encoder']}, "
      f"loss.w_cldice={reloaded['loss']['w_cldice']}")

# -- tests ----------------------------------------------------------------
check("pytest suite passed", pytest_run.returncode == 0,
      f"exit code {pytest_run.returncode}")
skipped = "skipped" in pytest_run.stdout.lower()
check("no test skipped for missing data", not skipped,
      "a skip here means the data was unreachable, not that the test passed"
      if skipped else "none skipped")

failed = [n for n, ok in checks if not ok]
print(f"\n{len(checks) - len(failed)}/{len(checks)} checks passed")
if failed:
    raise AssertionError("failed checks: " + ", ".join(failed))

## Push the measured configuration

Only `configs/default.yaml` changed here — `train.batch_size` was measured on
this host, and the `model:` and `loss:` sections were filled in on `main`
before this notebook ran. The model itself is code, and no checkpoint exists
yet. `expect` names the file, so a push that quietly finds nothing to commit is
diagnosed rather than reported as a no-op.

In [ ]:
from scripts.push_results import push_results

push_results(
    "step 5: unet model and compound loss",
    paths=PATHS,
    expect=[Path(PATHS["repo_root"]) / "configs" / "default.yaml"],
)